In [1]:
from embedder import Embedder

embed = Embedder()


2026-06-28 15:51:01.320737218 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


# Q1. Embedding a query

In [13]:
q1 = "How does approximate nearest neighbor search work?"

v1 = embed.encode(q1)

In [14]:
v1[0]

np.float64(-0.02058203437252893)

### Loading data

In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [6]:
len(documents)

72

# Q2. Cosine similarity

In [16]:
doc = next(d for d in documents if d['filename'] == "02-vector-search/lessons/07-sqlitesearch-vector.md")

In [17]:
q2 = doc['content']

v2 = embed.encode(q2)

In [22]:
v1 @ v2

np.float64(0.36107026789538205)

# Q3. Chunking and search by hand

In [23]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [25]:
texts = [chunk['content'] for chunk in chunks]

batch_size = 50
vectors = []

for i in range(0, len(texts), batch_size):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    vectors.extend(batch_vectors)

In [26]:
import numpy as np
X = np.array(vectors)

In [27]:
scores = X @ v1

In [28]:
idx = np.argmax(scores)
idx

np.int64(94)

In [32]:
scores[idx]

np.float64(0.6489016436447387)

In [29]:
chunks[idx]['filename']

'02-vector-search/lessons/07-sqlitesearch-vector.md'

# Q4. Vector search with minsearch

In [41]:
q4 = "What metric do we use to evaluate a search engine?"

v4 = embed.encode(q4)

In [42]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["content"])
vindex.fit(X, chunks)

In [43]:
results = vindex.search(
    v4,
    num_results=5
)

In [44]:
results[0]['filename']

'04-evaluation/lessons/05-search-metrics.md'

# Q5. Text search vs vector search

In [45]:
q5 = "How do I store vectors in PostgreSQL?"
v5 = embed.encode(q5)

In [46]:
v_results = vindex.search(
    v5,
    num_results=5
)

In [49]:
vector_set = set(result['filename'] for result in v_results)
vector_set

{'02-vector-search/lessons/08-pgvector.md',
 '03-orchestration/lessons/05-rag.md'}

In [50]:
from minsearch import Index

In [51]:
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(chunks)

In [52]:
t_results = index.search(
    q5,
    num_results=5
)

In [53]:
text_set = set(result['filename'] for result in t_results)
text_set

{'02-vector-search/lessons/01-intro.md',
 '02-vector-search/lessons/02-embeddings.md',
 '03-orchestration/lessons/05-rag.md'}

In [55]:
vector_set - text_set

{'02-vector-search/lessons/08-pgvector.md'}

# Q6. Hybrid search

In [56]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [57]:
q6 = "How do I give the model access to tools?"

v6 = embed.encode(q6)

In [58]:
vector_results = vindex.search(
    v6,
    num_results=5
)

In [59]:
text_results = index.search(
    q6,
    num_results=5
)

In [60]:
results = rrf([vector_results, text_results])

In [62]:
[result['filename'] for result in results][0]

'01-agentic-rag/lessons/13-function-calling.md'